# Entraînement LoRA de la Cascade S2 et Inférence S1/S2 (Étape 6)

Référence : `specifications.md` (sections 4.1, 4.5 et 5).
- En mode essai (`MODE_ESSAI = True`), ce notebook s'exécute de bout en bout sur 200 exemples factices (40 % D manuel, 30 % D écoute, 20 % P presse, 10 % D6 débuts), simule une coupure à l'étape 100, reprend jusqu'à l'étape 200, et génère l'exemple documenté de logits/softmax et de chat template.
- Sur Kaggle GPU (`MODE_ESSAI = False`), il charge `Qwen/Qwen3-4B` en 4 bits (`nf4`, `float16`) avec `peft.LoraConfig` (`r=16`, `alpha=32`, `dropout=0.05`).

In [ ]:
import os
from pathlib import Path
import sys
import tempfile

MODE_ESSAI = os.environ.get("MODE_ESSAI", "1") != "0"

# Localisation automatique de la racine du dépôt
racine = Path.cwd()
while not (racine / "configs" / "s2_cascade.yaml").exists() and racine.parent != racine:
    racine = racine.parent

dossier_cascade = racine / "code" / "cascade"
if str(dossier_cascade) not in sys.path:
    sys.path.insert(0, str(dossier_cascade))

from entrainement_lora import charger_config_s2, entrainer_lora_s2_essai
from inference_s1_s2 import executer_mode_essai_inference

cfg = charger_config_s2(racine / "configs" / "s2_cascade.yaml")
print("Système :", cfg["systeme"], "| Mode essai :", MODE_ESSAI)

In [ ]:
if MODE_ESSAI:
    with tempfile.TemporaryDirectory() as tmp_dir:
        # 1. Passe 1 avec coupure simulée à l'étape 100
        r1 = entrainer_lora_s2_essai(
            dossier_sortie=tmp_dir,
            config=cfg,
            nb_exemples=200,
            nb_etapes_max=200,
            sauvegarder_tous_les=50,
            interrompre_a_etape=100,
            reprendre=False,
        )
        # 2. Passe 2 avec reprise automatique de 100 à 200
        r2 = entrainer_lora_s2_essai(
            dossier_sortie=tmp_dir,
            config=cfg,
            nb_exemples=200,
            nb_etapes_max=200,
            sauvegarder_tous_les=50,
            interrompre_a_etape=None,
            reprendre=True,
        )
        res_inf = executer_mode_essai_inference(dossier_sortie=tmp_dir)
        print("Proportions 200 exemples :", r2["comptes_sources"])
        print("Reprise :", r2["reprise_effectuee"], "de", r2["etape_depart"], "à", r2["etape_finale"])
        print("Budget estimé :", r2["budget_estime"])
        print("Segments SRT générés :", len(res_inf["segments"]))